# Supervised Learning Basics
## Machine learning for energy applications — Module 1, Exercise 1

In this exercise we explore the foundational supervised learning algorithms that appear
throughout modern machine-learning workflows in the energy industry. We will work through
linear regression, regularization, K-nearest neighbours, decision trees, and random forests
using well-known reference datasets from scikit-learn.

**Evaluation habits used throughout.** We split the data before fitting anything. Models and
preprocessing steps are fitted on **training** rows only. Settings are chosen on **validation**
rows (or by cross-validation on the training rows), and every score table starts with a
**baseline**. The **test** rows are opened once, at the end, for the locked choice.

---
## 0 — Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing, get_data_home, load_iris
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    mean_absolute_error,
    r2_score,
    accuracy_score,
)

# Plotting defaults
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 12,
})

RANDOM_STATE = 42
print("Setup complete.")

---
## 1 — Linear Regression

We use the **California Housing** dataset, which contains median house values for
California districts derived from the 1990 census. Each sample has eight numerical
features (median income, house age, average rooms, etc.). The target is the median
house value in units of 100,000 dollars. A random split of the rows asks how well the
model predicts other districts from the same regions, which is the question for this
exercise; a model meant for a new region would need a spatially grouped split.

> **Data download.** `fetch_california_housing` needs internet access the first time it runs on a
> machine: it downloads the data (about 0.4 MB) and caches it in the scikit-learn data folder
> (`~/scikit_learn_data` by default, or the folder named by the `SCIKIT_LEARN_DATA` environment
> variable). Offline machines need a pre-filled cache: copy that folder, which then holds
> `cal_housing_py3.pkz`, from a machine where this cell has run. Without internet access or a
> cache, the loading cell stops with an error that says so.

### 1.1 Load Data and Fit a Linear Model

In [ ]:
# YOUR CODE HERE
# TODO: Load the California Housing dataset using fetch_california_housing(as_frame=True)
#       (the first call downloads it; see the note above if it fails)
# TODO: Store features in X_housing and target in y_housing (units: $100,000)
# TODO: Print the feature names, number of samples, and number of features
# TODO: Display the first few rows with .head()

In [ ]:
# YOUR CODE HERE
# TODO: Split first, before fitting anything: hold out 20% of the rows
#       (test_size=0.2, random_state=RANDOM_STATE), then split the held-out rows in half
#       (test_size=0.5, random_state=RANDOM_STATE) into validation and test rows.
#       Keep the test rows closed until section 2.3.
# TODO: Fit DummyRegressor() (the mean baseline) and a LinearRegression on the training rows
# TODO: Build a score table with training R², validation R² and validation MAE,
#       baseline row first

### 1.2 Polynomial Features

Adding polynomial features (degree 2) allows the linear model to capture non-linear
relationships. We scale the data first to keep polynomial terms numerically stable.
The scaler learns means and standard deviations, so it is fitted on the training rows
only; validation rows are only transformed with those training statistics.

In [ ]:
# YOUR CODE HERE
# TODO: Fit a StandardScaler on the training rows only; transform training and validation rows
# TODO: Create degree-2 polynomial features using PolynomialFeatures(degree=2, include_bias=False)
# TODO: Fit a LinearRegression on the polynomial features of the training rows
# TODO: Print the number of polynomial features and add a row to your score table

### 1.3 Demonstrating Overfitting with High-Degree Polynomials

We use a *single feature* (MedInc) so we can visualize the effect of increasing
polynomial degree, and compare training and validation R² for degrees 1, 2, 5 and 15.
We fit each degree twice: with raw MedInc powers, and with the polynomial columns
standardized, `make_pipeline(PolynomialFeatures(d), StandardScaler(), LinearRegression())`.

A bigger polynomial contains every smaller one, so **training** R² can never fall as the
degree grows. If it does, the least-squares solver has failed on badly scaled columns
(raw MedInc¹⁵ reaches about 15¹⁵ ≈ 4 × 10¹⁷), not the model. How badly it fails depends
on the scikit-learn version; with the columns standardized the fit is stable. With
16,512 training rows and one feature there is little room to overfit at all.

In [ ]:
# YOUR CODE HERE
# TODO: Extract the "MedInc" feature from the training and validation rows
# TODO: For degrees [1, 2, 5, 15], fit two pipelines on the training rows:
#       make_pipeline(PolynomialFeatures(d, include_bias=False), LinearRegression())
#       make_pipeline(PolynomialFeatures(d, include_bias=False), StandardScaler(), LinearRegression())
# TODO: Compute training and validation R², and plot each fitted curve over the validation rows
#       in a 2x4 subplot figure (one row per variant)
# Hint: use np.linspace to create a smooth grid for plotting the fitted curve

---
## 2 — Regularization: Ridge (L2) and Lasso (L1)

Regularization adds a penalty to the loss function to prevent large coefficients,
reducing overfitting.

| Method | Penalty | Effect |
|--------|---------|--------|
| **Ridge (L2)** | Sum of squared coefficients | Shrinks coefficients toward zero |
| **Lasso (L1)** | Sum of absolute coefficients | Drives some coefficients exactly to zero (feature selection) |

We compare α values on the **validation** rows; section 2.3 locks one choice and opens the
test rows once.

### 2.1 Ridge vs Lasso on Polynomial Features

In [ ]:
# YOUR CODE HERE
# TODO: For alphas = [0.01, 0.1, 1.0, 10.0, 100.0], fit both Ridge and Lasso
#       on the degree-2 polynomial features of the training rows (X_train_poly2)
# TODO: For each alpha, record the validation R² (X_val_poly2) and the number of
#       non-zero coefficients
# TODO: Display results in a pandas DataFrame
# Hint: use Lasso(alpha=alpha, max_iter=10000) to ensure convergence

### 2.2 Coefficient Comparison — Lasso Feature Selection

In [ ]:
# YOUR CODE HERE
# TODO: Fit Ridge and Lasso with alpha=0.1 on the degree-2 polynomial features
# TODO: Create a 1x2 subplot figure
#       - Left panel: bar chart of Ridge coefficients
#       - Right panel: bar chart of Lasso coefficients
# TODO: Print how many coefficients Lasso zeroed out


### 2.3 Lock the Choice, Then Open the Test Rows Once

We choose on validation rows only: the model and α with the highest validation R² in the
table above. Differences of a few thousandths are within the noise of a single validation
split, so preferring the more regularized α within such a plateau is also defensible; what
matters is that the choice is made before the test rows are used. Because we chose on the validation rows,
their best score is optimistic. The test rows have not been used for anything yet: we
score the locked model on them once, next to the baseline on the same rows. Changing the
model after seeing this score would turn the test rows into validation rows.

In [ ]:
# YOUR CODE HERE
# TODO: Lock one choice: the model and alpha with the highest validation R²
# TODO: Transform the test rows with the scaler and polynomial step fitted on the training rows
# TODO: Score the locked model and the mean baseline on the test rows, once, and print both

---
## 3 — K-Nearest Neighbours (KNN)

KNN is a simple, instance-based algorithm: to classify a new point, find the *K* closest
training points and take a majority vote. We use the **Iris** dataset (150 samples,
4 features, 3 classes).

Iris is small: after holding out 30 test rows, 120 training rows remain, and one row of a
30-row set is worth 3.3 accuracy points. A single validation split would be too noisy to
compare settings, so we compare them with **5-fold stratified cross-validation on the
training rows** (the same folds for every model) and keep the test rows closed until
section 4.3.

### 3.1 Load Iris Data and Train KNN (K=5)

In [ ]:
# YOUR CODE HERE
# TODO: Load the Iris dataset using load_iris(as_frame=True); print the label encoding,
#       dict(enumerate(iris.target_names.tolist()))
# TODO: Split into training and test rows (test_size=0.2, random_state=RANDOM_STATE);
#       keep the test rows closed until section 4.3
# TODO: Create cv_iris = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
# TODO: Print the 5-fold CV accuracy (cross_val_score on the training rows) of
#       DummyClassifier(strategy="most_frequent") and of KNeighborsClassifier(n_neighbors=5)

### 3.2 Accuracy vs K

In [ ]:
# YOUR CODE HERE
# TODO: For k_values = [1, 3, 5, 10, 20], record training accuracy and 5-fold CV accuracy
#       (mean and SD, same cv_iris folds)
# TODO: Plot both against K, with the majority-class baseline as a horizontal line
# TODO: Print the results for each K value

---
## 4 — Decision Tree

Decision trees recursively partition the feature space using axis-aligned splits.
They are easy to interpret but prone to overfitting if not constrained.

### 4.1 Train and Visualize

In [ ]:
# YOUR CODE HERE
# TODO: Train a DecisionTreeClassifier with no depth limit on the Iris training rows
# TODO: Print its 5-fold CV accuracy (same cv_iris folds) and the tree depth
# TODO: Visualize the tree using plot_tree with feature_names and class_names
# Hint: use filled=True and rounded=True for a cleaner visualization

### 4.2 Effect of `max_depth`

In [ ]:
# YOUR CODE HERE
# TODO: For depths = [1, 2, 3, 5, None], compute the 5-fold CV accuracy of a
#       DecisionTreeClassifier and fit it on the training rows
# TODO: Create a 1x5 subplot figure showing each tree with its CV accuracy
# Hint: use plot_tree in each subplot

### 4.3 Lock One Classifier, Then Open the Test Rows Once

The candidates are the K values of section 3.2 and the depths of section 4.2, compared on
the same cross-validation folds. We lock the one with the highest mean CV accuracy (the
first listed wins a tie), fit it on all training rows, and score the 30 test rows once.
With so few rows, the test score can confirm a choice but could not have ranked these
candidates.

In [ ]:
# YOUR CODE HERE
# TODO: Lock one classifier: the K or max_depth with the highest mean CV accuracy
# TODO: Fit it on all training rows and score the test rows once; print the number of
#       correct rows out of 30 and the majority-class baseline on the same test rows

---
## 5 — Random Forest

A **Random Forest** is an ensemble of decision trees trained on bootstrapped samples
with random feature subsets. This reduces variance and typically outperforms a single
tree.

The test rows were used once, for the locked choice in section 4.3, so the forest is
compared with the other models by cross-validation on the same training folds only.
(Had it been a candidate, it would have joined the section 4.3 comparison before the test.)

### 5.1 Train and Evaluate

In [ ]:
# YOUR CODE HERE
# TODO: Print the 5-fold CV accuracy of RandomForestClassifier(n_estimators=100,
#       random_state=RANDOM_STATE) on the training rows (same cv_iris folds)
# TODO: Fit it on all training rows (for the feature importances below)

### 5.2 Feature Importance

Impurity-based importances describe how this forest uses the features on the training
rows; they are not causal effects. Module 2 compares them with permutation importance.

In [ ]:
# YOUR CODE HERE
# TODO: Extract feature_importances_ from the trained random forest
# TODO: Create a horizontal bar chart showing feature importance sorted ascending
# TODO: Print each feature name with its importance value


---
## Summary

In this exercise we covered:

1. **Linear Regression** — fitting, evaluation metrics (R², MAE), polynomial features, and overfitting, with a mean baseline first in the score table.
2. **Regularization** — Ridge (L2) and Lasso (L1), with Lasso performing automatic feature selection; α chosen on validation rows and the locked model tested once.
3. **K-Nearest Neighbours** — instance-based classification and the effect of K on bias-variance trade-off, compared by cross-validation against a majority-class baseline.
4. **Decision Trees** — interpretable models and the role of `max_depth` in controlling complexity; one locked classifier scored once on the test rows.
5. **Random Forest** — ensemble learning and feature importance.

These algorithms form the building blocks for more advanced techniques used across
the energy industry, from production forecasting to well-log analysis. In those settings,
the split must also follow the deployment question: whole wells or later time periods,
not random rows.